# Chapter 7: Applications in Machine Learning & AI

This capstone chapter shows how every topic from Chapters 1–6 appears in real ML systems.

| Topic | Linear Algebra Used |
|---|---|
| PCA | Eigendecomposition of covariance matrix |
| Linear Regression | Normal equations: $(X^TX)^{-1}X^Ty$ |
| Recommendation Systems | SVD / matrix factorisation |
| NLP Similarity | Cosine similarity of embedding vectors |
| Neural Networks | Matrix multiply + transpose at every layer |
| Latent Semantic Analysis | SVD of term-document matrix |

## Learning Objectives
- Implement PCA from scratch and compare with scikit-learn
- Derive and solve linear regression via normal equations
- Build a matrix factorisation recommender
- Compute cosine similarity and visualise an embedding space
- Trace the exact matrix shapes through a neural network forward pass


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)


## 1. Principal Component Analysis (PCA)

PCA finds the directions of **maximum variance** in data.

**Algorithm:**
1. Center the data: $\tilde{X} = X - \bar{X}$
2. Compute covariance: $C = \frac{1}{n-1}\tilde{X}^T\tilde{X}$
3. Eigendecompose: $C = Q\Lambda Q^T$
4. Project: $Z = \tilde{X} Q_k$ (keep top $k$ eigenvectors)

**Explained variance ratio** of component $i$: $\lambda_i / \sum_j \lambda_j$


In [ ]:
np.random.seed(7)

# Correlated 2D data
mean = [2, 3]
cov  = [[4, 3], [3, 3]]
X_2d = np.random.multivariate_normal(mean, cov, 200)

# PCA from scratch
X_c = X_2d - X_2d.mean(axis=0)
C   = X_c.T @ X_c / (len(X_c) - 1)
vals_pca, vecs_pca = np.linalg.eigh(C)
idx = np.argsort(vals_pca)[::-1]
vals_pca, vecs_pca = vals_pca[idx], vecs_pca[:, idx]
explained = vals_pca / vals_pca.sum() * 100

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(*X_2d.T, alpha=0.4, s=20)
ax.scatter(*X_2d.mean(axis=0), c='red', s=100, zorder=5)
for i, (lam, vec) in enumerate(zip(vals_pca, vecs_pca.T)):
    scale = np.sqrt(lam) * 2
    ax.annotate('', xy=X_2d.mean(axis=0) + scale*vec,
                xytext=X_2d.mean(axis=0) - scale*vec,
                arrowprops=dict(arrowstyle='<->', color='red', lw=3))
    ax.text(*(X_2d.mean(axis=0) + scale*vec*1.1),
            f'PC{i+1}\n{explained[i]:.1f}%', color='red', fontsize=10, ha='center')
ax.set(aspect='equal', title='PCA: Principal Component Arrows')
plt.tight_layout(); plt.show()
print("Explained variance:", [f"{e:.1f}%" for e in explained])


In [ ]:
# PCA on Iris (4D → 2D)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
species = ['Setosa', 'Versicolor', 'Virginica']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# From scratch
C_iris = X_scaled.T @ X_scaled / (len(X_scaled)-1)
vals_i, vecs_i = np.linalg.eigh(C_iris)
idx_i = np.argsort(vals_i)[::-1]
vecs_i = vecs_i[:, idx_i]; vals_i = vals_i[idx_i]
Z_scratch = X_scaled @ vecs_i[:, :2]

# sklearn
pca_sk = PCA(n_components=2)
Z_sk = pca_sk.fit_transform(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for sp_idx, (sp, col) in enumerate(zip(species, colors)):
    mask = y_iris == sp_idx
    ax1.scatter(*Z_scratch[mask].T, c=col, label=sp, alpha=0.7, s=30)
    ax2.scatter(*Z_sk[mask].T,      c=col, label=sp, alpha=0.7, s=30)
ax1.set(title='PCA from Scratch', xlabel='PC1', ylabel='PC2'); ax1.legend()
ax2.set(title='sklearn PCA', xlabel='PC1', ylabel='PC2'); ax2.legend()
plt.suptitle('Iris Dataset: PCA 4D → 2D', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print("Explained variance ratios:", pca_sk.explained_variance_ratio_.round(3))


## 2. Linear Regression via Normal Equations

Model: $y = X\beta + \epsilon$ where $X \in \mathbb{R}^{n \times p}$ includes a bias column.

**Closed-form solution** (minimises MSE analytically):

$$\hat{\beta} = (X^T X)^{-1} X^T y$$

This is the unique minimum of $\|y - X\beta\|^2$ — no gradient descent needed!
sklearn's `LinearRegression` computes this internally via SVD for numerical stability.


In [ ]:
np.random.seed(0)
n = 100
x_lr = np.linspace(0, 10, n)
y_lr = 3.0 * x_lr + 2.0 + np.random.randn(n) * 2.5

# Augment with bias
X_lr = np.column_stack([x_lr, np.ones(n)])

# Normal equations
beta_hat = np.linalg.inv(X_lr.T @ X_lr) @ X_lr.T @ y_lr
print(f"Normal equations: slope={beta_hat[0]:.4f}, intercept={beta_hat[1]:.4f}")

# sklearn comparison
lr = LinearRegression().fit(x_lr.reshape(-1,1), y_lr)
print(f"sklearn:          slope={lr.coef_[0]:.4f}, intercept={lr.intercept_:.4f}")

y_pred = X_lr @ beta_hat
ss_res = np.sum((y_lr - y_pred)**2)
ss_tot = np.sum((y_lr - y_lr.mean())**2)
r2     = 1 - ss_res / ss_tot
print(f"R² = {r2:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(x_lr, y_lr, alpha=0.5, s=20, label='Data')
ax1.plot(x_lr, y_pred, 'r', lw=2, label=f'Fit: {beta_hat[0]:.2f}x + {beta_hat[1]:.2f}')
ax1.set(title=f'Linear Regression (R²={r2:.3f})', xlabel='x', ylabel='y'); ax1.legend()

residuals = y_lr - y_pred
ax2.stem(x_lr, residuals, markerfmt='C0o', linefmt='C0-', basefmt='k-')
ax2.axhline(0, color='r', lw=2); ax2.set(title='Residuals', xlabel='x', ylabel='y - ŷ')
plt.tight_layout(); plt.show()


## 3. Matrix Factorisation for Recommendations

A **user-item rating matrix** $R \approx UV^T$ can be approximated by a low-rank factorisation.

SVD finds the **optimal** rank-$k$ approximation (Eckart-Young theorem).
This is the mathematical foundation of Netflix-style **collaborative filtering**.

Missing entries (unrated items) are filled by the reconstructed matrix $\hat{R} = U\Sigma V^T$.


In [ ]:
# 5 users × 6 items rating matrix (1-5 scale, NaN = unrated)
R = np.array([
    [5,   3,   np.nan, 1,   np.nan, 4  ],
    [4,   np.nan, 4,   1,   2,      np.nan],
    [np.nan, 1,  5,   np.nan, 5,   2  ],
    [1,   1,   np.nan, 4,   4,      np.nan],
    [np.nan, 2,  4,   4,   np.nan, 3  ],
])
user_labels  = [f'User {i+1}'  for i in range(5)]
item_labels  = [f'Item {j+1}'  for j in range(6)]

# Fill missing with column means
R_filled = R.copy()
col_means = np.nanmean(R, axis=0)
for j in range(R.shape[1]):
    R_filled[np.isnan(R[:, j]), j] = col_means[j]

# SVD rank-2 approximation
U_r, s_r, Vt_r = np.linalg.svd(R_filled, full_matrices=False)
k_rec = 2
R_hat = U_r[:, :k_rec] @ np.diag(s_r[:k_rec]) @ Vt_r[:k_rec, :]
R_hat = np.clip(R_hat, 1, 5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.heatmap(R, annot=True, fmt='.1f', xticklabels=item_labels,
            yticklabels=user_labels, cmap='YlOrRd', ax=axes[0], vmin=1, vmax=5)
axes[0].set_title('Original Ratings (NaN = unrated)')
sns.heatmap(R_hat, annot=True, fmt='.1f', xticklabels=item_labels,
            yticklabels=user_labels, cmap='YlOrRd', ax=axes[1], vmin=1, vmax=5)
axes[1].set_title('Reconstructed (predicted) Ratings')
plt.tight_layout(); plt.show()

print("\nTop recommended item for each user:")
for i, u in enumerate(user_labels):
    unrated = np.where(np.isnan(R[i]))[0]
    if len(unrated) > 0:
        best = unrated[np.argmax(R_hat[i, unrated])]
        print(f"  {u}: {item_labels[best]} (predicted={R_hat[i,best]:.2f})")


## 4. Cosine Similarity in NLP

Word and document **embeddings** are vectors in high-dimensional space.
Semantic similarity ≈ **cosine of the angle** between vectors:

$$\text{sim}(u, v) = \frac{u \cdot v}{\|u\| \|v\|} = \cos\theta \in [-1, 1]$$

- $+1$ → identical direction (very similar)
- $0$ → orthogonal (unrelated)
- $-1$ → opposite direction (antonyms)

Used in: semantic search, clustering, RAG (retrieval-augmented generation), duplicate detection.


In [ ]:
np.random.seed(3)

words = ['king', 'queen', 'man', 'woman', 'dog', 'cat', 'car', 'truck']
# Random 4D embeddings (in practice these come from Word2Vec/BERT/etc.)
embeddings = np.random.randn(8, 4)
# Normalise to unit vectors
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Cosine similarity matrix
sim_matrix = embeddings @ embeddings.T  # since ||e||=1, dot product = cosine sim

fig, ax = plt.subplots(figsize=(8, 7))
mask = np.eye(len(words), dtype=bool)
sns.heatmap(sim_matrix, annot=True, fmt='.2f',
            xticklabels=words, yticklabels=words,
            cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Cosine Similarity Matrix of Word Embeddings', fontsize=13)
plt.tight_layout(); plt.show()

# Top-3 similar pairs (excluding diagonal)
sim_flat = sim_matrix.copy()
np.fill_diagonal(sim_flat, -np.inf)
print("\nTop-3 most similar pairs:")
for _ in range(3):
    i, j = np.unravel_index(np.argmax(sim_flat), sim_flat.shape)
    print(f"  {words[i]} ↔ {words[j]}: {sim_matrix[i,j]:.3f}")
    sim_flat[i, j] = sim_flat[j, i] = -np.inf


## 5. Neural Networks as Matrix Operations

A **dense (fully connected) layer** is:

$$h = \sigma(Wx + b)$$

For an entire **mini-batch** $X \in \mathbb{R}^{n \times d_{in}}$:

$$H = \sigma(X W^T + b)$$

The **entire forward pass** of a deep network is a sequence of matrix multiplications and activations.
The **backward pass** uses transposes and the chain rule — more matrix operations.

> **Linear algebra IS deep learning.**


In [ ]:
# 2-layer network: input(4) → hidden(8) → output(3)
np.random.seed(0)

def relu(x): return np.maximum(0, x)
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# Weight matrices (Xavier initialisation)
W1 = np.random.randn(8, 4) * np.sqrt(2/4)   # (hidden, input)
b1 = np.zeros(8)
W2 = np.random.randn(3, 8) * np.sqrt(2/8)   # (output, hidden)
b2 = np.zeros(3)

# Mini-batch of 5 samples with 4 features each
X_nn = np.random.randn(5, 4)   # (batch, input)

print("=== Forward Pass Shape Trace ===")
print(f"Input X:          {X_nn.shape}")

Z1 = X_nn @ W1.T + b1          # (5,4) @ (4,8) + (8,) = (5,8)
print(f"Z1 = X @ W1^T + b1: {Z1.shape}  [{X_nn.shape} @ {W1.T.shape}]")

H1 = relu(Z1)
print(f"H1 = ReLU(Z1):     {H1.shape}")

Z2 = H1 @ W2.T + b2            # (5,8) @ (8,3) + (3,) = (5,3)
print(f"Z2 = H1 @ W2^T + b2:{Z2.shape}  [{H1.shape} @ {W2.T.shape}]")

H2 = softmax(Z2)
print(f"Output = softmax:  {H2.shape}")
print("\nOutput (class probabilities for 5 samples):\n", np.round(H2, 3))
print("Row sums (should be 1):", H2.sum(axis=1).round(4))


## 6. Latent Semantic Analysis (LSA)

**LSA** applies SVD to a **term-document matrix** to discover hidden (latent) topics.
Words and documents are embedded in a low-dimensional semantic space.
Documents about similar topics cluster together.

The term-document matrix $M \in \mathbb{R}^{t \times d}$ has:
- Rows = terms (words)
- Columns = documents
- Entries = (TF-IDF) word frequencies

$M = U\Sigma V^T$ → keep top-$k$ → latent semantic space.


In [ ]:
# Small 6-word × 5-document term-document matrix
terms = ['python', 'matrix', 'neural', 'network', 'algebra', 'linear']
docs  = ['ML intro', 'Deep Learning', 'Lin. Algebra', 'Data Science', 'NLP']

# Each column = document word count vector
M_lsa = np.array([
    [3, 2, 0, 2, 1],  # python
    [1, 0, 3, 0, 1],  # matrix
    [0, 4, 0, 1, 2],  # neural
    [0, 3, 0, 1, 2],  # network
    [2, 0, 4, 0, 1],  # algebra
    [2, 0, 3, 1, 1],  # linear
], dtype=float)

U_l, s_l, Vt_l = np.linalg.svd(M_lsa, full_matrices=False)

# Embed documents and terms in rank-2 latent space
doc_coords  = np.diag(s_l[:2]) @ Vt_l[:2, :]   # (2, n_docs)
term_coords = U_l[:, :2]                          # (n_terms, 2)

fig, ax = plt.subplots(figsize=(8, 6))
# Documents
ax.scatter(*doc_coords, s=100, c='steelblue', zorder=5)
for i, doc in enumerate(docs):
    ax.annotate(doc, doc_coords[:, i], fontsize=10,
                textcoords='offset points', xytext=(6, 4), color='steelblue')
# Terms
ax.scatter(*term_coords.T, s=80, c='tomato', marker='^', zorder=5)
for i, term in enumerate(terms):
    ax.annotate(term, term_coords[i], fontsize=10,
                textcoords='offset points', xytext=(6, -8), color='tomato')

ax.axhline(0, 'k', lw=0.5); ax.axvline(0, 'k', lw=0.5)
ax.set(title='Latent Semantic Analysis: Documents & Terms in 2D Topic Space',
       xlabel='Latent Dimension 1', ylabel='Latent Dimension 2')
plt.tight_layout(); plt.show()


## Summary & What's Next

Congratulations! You have completed the core of linear algebra for ML:

| Chapter | Topic | Key Takeaway |
|---|---|---|
| 1 | Vectors | Ordered lists with geometry |
| 2 | Matrices | Represent data and transformations |
| 3 | Linear Systems | Every ML model solves $Ax=b$ |
| 4 | Vector Spaces | Geometry of solutions |
| 5 | Eigenvalues | Intrinsic structure of matrices |
| 6 | Decompositions | Factor → understand → compute |
| 7 | ML Applications | Everything connects here |

## Recommended Next Steps

1. **Calculus for ML** — gradients, Jacobians, Hessians, backpropagation
2. **Probability & Statistics** — Bayes' theorem, distributions, MLE
3. **Deep Learning** — PyTorch/JAX, automatic differentiation
4. **Advanced LA** — iterative solvers, randomised SVD, tensor decompositions

> "The purpose of computing is insight, not numbers." — Richard Hamming

Thank you for watching! If this series helped you, please like and subscribe.
